# Load the three datasets
Downloads HelpSteer2, PKU-SafeRLHF and UltraFeedback from Hugging Face and puts them in the same format
(`prompt`, `chosen`, `rejected`). No GPU needed. The first run takes a couple of minutes.

In [1]:
# get the latest code from GitHub.
!test -d /content/mfr-dpo || git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
!git -C /content/mfr-dpo pull -q
%cd /content/mfr-dpo

/content/mfr-dpo


In [3]:
import os, sys
sys.path.insert(0, "/content/mfr-dpo/src" if os.path.exists("/content/mfr-dpo") else "../src")
from mfr_data import load_helpsteer2, load_pku_saferlhf, load_ultrafeedback

In [4]:
helpful = load_helpsteer2()
safe = load_pku_saferlhf()
quality = load_ultrafeedback()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/25.0k [00:00<?, ?B/s]

preference/preference.jsonl.gz: reconstructing file:   0%|          |  0.00B / 15.3MB            

preference/preference.jsonl.gz: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

...

Generating train_sft split:   0%|          | 0/61135 [00:00<?, ? examples/s]

Generating test_prefs split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/61135 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [6]:
import pandas as pd

summary = pd.DataFrame({
    name: {
        "pairs": len(df),
        "unique prompts": df["prompt"].nunique(),
        "avg prompt chars": int(df["prompt"].str.len().mean()),
        "avg chosen chars": int(df["chosen"].str.len().mean()),
        "avg rejected chars": int(df["rejected"].str.len().mean()),
    }
    for name, df in {"helpful": helpful, "safe": safe, "quality": quality}.items()
})

In [8]:
print(summary)

                    helpful   safe  quality
pairs                  2765  10796    42182
unique prompts         2762   8555    42174
avg prompt chars        390    129      671
avg chosen chars       1463    455     1239
avg rejected chars     1348    528     1003


In [9]:
i = 0
for name, df in {"helpful": helpful, "safe": safe, "quality": quality}.items():
    row = df.iloc[i]
    print(f"=============== {name} ===============")
    print("PROMPT:  ", row["prompt"][:500])
    print("\nCHOSEN:  ", row["chosen"][:500])
    print("\nREJECTED:", row["rejected"][:500], "\n")

=============== helpful ===============
PROMPT:   Please make a list of independent Fertility coaching and consulting services in USA

CHOSEN:   Sure, here is a list of independent Fertility coaching and consulting services in USA:

1. Fertility Focus LLC
2. Fertility Journey Inc.
3. Fertility Road LLC
4. Fertility Wellness LLC
5. The Fertility Coach LLC
6. Fertility Consulting Services LLC
7. Fertility Health Services LLC
8. Fertility Support Services LLC
9. Fertility Advocates LLC
10. Fertility Resource Group LLC
11. Fertility Solutions LLC
12. Fertility Consulting LLC
13. Fertility Advocates and Solutions LLC
14. Fertility Advocates a

REJECTED: Sure, here are some independent Fertility coaching and consulting services in the USA:

1. Fertility Authority
2. Fertility Solutions
3. Fertility Success Coaching
4. Fertility Consulting Services
5. Fertility Coaching and Consulting
6. Fertility Coaching and Consulting Services
7. Fertility Coaching and Consulting Services
8. Fertility Coac

In [10]:
helpful.head()

,prompt,chosen,rejected
0,Please make a list of independent Fertility co...,"Sure, here is a list of independent Fertility ...","Sure, here are some independent Fertility coac..."
1,A wallet provider offers a wallet that can't c...,This wallet would be considered a semi-closed ...,This wallet is a closed wallet because it can ...
2,you are a restaurant expansion consultant. I w...,"Sure, I can help you with that! Here's a promp...","As a restaurant expansion consultant, my goal ..."
3,Can Artificial Intelligence be a threath for E...,Artificial Intelligence (AI) has the potential...,Artificial intelligence (AI) is a rapidly grow...
4,hiGoodBad 1 words 2 char Copy Text Copy ...,Hi there! It's nice to see you. How can I help...,Here are some interesting facts about the Engl...
